# Final LLM Eval: base vs sft vs reference

**Input:** `final_llm_result.jsonl` (150 rows: `id, question, reference_answer, base_answer, sft_answer, task_type, grade, subject`)

**Pipeline:** A) offline auto metrics (ROUGE/BLEU/chrF/METEOR/BERTScore/embeddings + numeric/MCQ exact-match)


> Run cells top-to-bottom. Works in Colab and local Jupyter. Set `GEMINI_API_KEY` when asked.

In [ ]:
# 0) Install deps (Colab: runs once; local Jupyter: safe to re-run)
!pip install -q rouge-score sacrebleu nltk bert-score sentence-transformers pandas numpy tqdm matplotlib scipy google-generativeai
print('deps installed')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 8.5 MB/s eta 0:00:00
deps installed


In [ ]:
# 0b) GPU check -- Colab: Runtime -> Change runtime type -> T4 GPU BEFORE running
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} device={DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} VRAM={torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")
else:
    print("WARNING: no CUDA -- BERTScore roberta-large will be very slow on CPU. Switch to T4 GPU runtime.")


torch 2.11.0+cu128 device=cuda
GPU: Tesla T4 VRAM=15.6GB


In [ ]:
# 1) Config — FULL-TEXT MODE (no truncation for judge or lexical metrics)
import os
MODEL_NAME = "gemini-3.5-flash-lite"  # exact model from your quota screen; if API returns 404, try "gemini-2.0-flash-lite"
FALLBACK_MODEL = "gemini-2.0-flash-lite"
DELAY_SEC = 4.5        # 60/4.5 = ~13 RPM, safe margin under 15 RPM
MAX_RETRIES = 4
MAX_OUTPUT_TOKENS = 600
TEMPERATURE = 0.0
FULL_TEXT_MODE = True  # judge + ROUGE/BLEU/chrF/METEOR all get full question/ref/base/sft
CAPS = {'question': 50000, 'reference': 50000, 'answer': 50000}  # effectively off; kept only as safety net
# NOTE: worst row in final_llm_result.jsonl is 10,634 chars total (~2.7k tokens) + ~800 prompt = ~3.5k in/row.
# At ~13 RPM -> ~45k TPM << 250K limit. 150 rows = 150 RPD << 500. Full-text is safe.

# Input: local mac path first, Colab /content fallback, else manual upload
CANDIDATE_INPUTS = [
    "/Users/varunesh/Desktop/Sargvision/final_llm_result.jsonl",
    "/content/final_llm_result.jsonl",
    "final_llm_result.jsonl",
]
INPUT_PATH = next((p for p in CANDIDATE_INPUTS if os.path.exists(p)), CANDIDATE_INPUTS[0])
OUTDIR = "/content" if os.path.exists("/content/drive") or os.path.basename(os.getcwd()) == "content" else "."
AUTO_OUT = os.path.join(OUTDIR, "eval_auto_scored.jsonl")
JUDGE_OUT = os.path.join(OUTDIR, "judge_checkpoint.jsonl")
FINAL_OUT = os.path.join(OUTDIR, "final_llm_eval_scored.jsonl")
print(f"MODEL={MODEL_NAME} INPUT={INPUT_PATH} OUTDIR={OUTDIR} FULL_TEXT={FULL_TEXT_MODE}")


MODEL=gemini-3.5-flash-lite INPUT=/content/final_llm_result.jsonl OUTDIR=/content FULL_TEXT=True


In [ ]:
# 2) Load + audit data
import json, collections, statistics
rows = []
with open(INPUT_PATH) as f:
    for i, line in enumerate(f, 1):
        line = line.strip()
        if line:
            rows.append(json.loads(line))
print(f"n={len(rows)} keys={sorted(rows[0].keys())}")
print(collections.Counter(r.get('task_type') for r in rows))
print(collections.Counter(r.get('subject') for r in rows).most_common(10))
missing = sum(1 for r in rows if not r.get('base_answer') or not r.get('sft_answer') or not r.get('reference_answer'))
print(f"rows with any missing answer field: {missing}")
tot = [len(r.get('question',''))+len(r.get('reference_answer',''))+len(r.get('base_answer',''))+len(r.get('sft_answer','')) for r in rows]
tot.sort()
print(f"total-chars mean={sum(tot)//len(tot)} p50={tot[len(tot)//2]} p95={tot[int(len(tot)*0.95)]} max={max(tot)}")
print(f"est input tokens total ~{sum(tot)//4} (+prompt overhead ~150*800)")

n=150 keys=['base_answer', 'grade', 'id', 'question', 'reference_answer', 'sft_answer', 'subject', 'task_type']
Counter({'practice_question': 58, 'other': 42, 'explain': 14, 'lesson_plan': 12, 'report': 7, 'feedback': 7, 'mcq': 7, 'numerical': 3})
[('Mathematics', 20), ('Science', 20), ('EVS', 20), ('Social Science', 20), ('Biology', 15), ('Geography', 15), ('Economics', 14), ('Chemistry', 13), ('Physics', 13)]
rows with any missing answer field: 0
total-chars mean=4926 p50=4601 p95=9395 max=10634
est input tokens total ~184728 (+prompt overhead ~150*800)


## Part A — Offline auto metrics (0 quota, FULL TEXT)
`ROUGE-1/2/L + sacreBLEU + chrF + METEOR` run on **full** question/ref/answers (no truncation).
`BERTScore-F1 (roberta-large) + MiniLM cosine` also receive full strings, but note their encoders internally cap at ~512 tokens — tails beyond that are down-weighted by the model itself. Winner per row on `BERTScore + ROUGE-L` composite; tie if delta < 0.02.
`Gemini judge (Part B)` receives full text with no caps.


In [ ]:
# 3) Metric helpers
import re, string
import nltk
nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
try:
    nltk.download('punkt', quiet=True)
except Exception:
    pass
from rouge_score import rouge_scorer
import sacrebleu
rouge = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

def trunc(s, n):
    s = (s or '').strip()
    return s if len(s) <= n else s[:n] + ' …[truncated]'

def safe_meteor(ref, hyp):
    try:
        return float(meteor_score([ref.split()], hyp.split()))
    except Exception:
        return 0.0

def num_extract(s):
    """Extract last numeric answer (handles units, fractions)."""
    if not s: return None
    ms = re.findall(r'-?\d+(?:\.\d+)?', s.replace(',', ''))
    return float(ms[-1]) if ms else None

def mcq_extract(s):
    if not s: return None
    m = re.search(r'\b([A-D])\b', s)
    return m.group(1) if m else None

print('helpers ready')

helpers ready


In [ ]:
# 4) Run auto metrics (batched BERTScore + embeddings, row-wise lexical) -- T4 GPU aware
import json, time
from tqdm import tqdm
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BSZ = 64 if DEVICE == "cuda" else 8  # T4 16GB safe for full-text; CPU fallback tiny batches
print(f"[Part A] device={DEVICE} bert_batch={BSZ} rows={len(rows)}", flush=True)
t_all = time.time()
refs = [r.get('reference_answer','') for r in rows]
bases = [r.get('base_answer','') for r in rows]
sfts = [r.get('sft_answer','') for r in rows]
print(f"[Part A] starting auto metrics on {len(rows)} rows...", flush=True)

# --- Stage 1/3: BERTScore (batch, verbose shows progress) ---
print(f"[1/3] BERTScore roberta-large on {DEVICE} (batch={BSZ}): base vs ref ...", flush=True)
t0 = time.time()
try:
    from bert_score import score as bert_score
    _, _, Fb_base = bert_score(bases, refs, model_type='roberta-large', device=DEVICE, batch_size=BSZ, verbose=True)
    print(f"[1/3] BERTScore roberta-large on {DEVICE}: sft vs ref ...", flush=True)
    _, _, Fb_sft = bert_score(sfts, refs, model_type='roberta-large', device=DEVICE, batch_size=BSZ, verbose=True)
    bert_base = [float(x) for x in Fb_base]; bert_sft = [float(x) for x in Fb_sft]
    print(f"[1/3] BERTScore done in {time.time()-t0:.1f}s", flush=True)
except Exception as e:
    print(f"[1/3] BERTScore skipped: {e}", flush=True)
    bert_base = [0.0]*len(rows); bert_sft = [0.0]*len(rows)

# --- Stage 2/3: MiniLM cosine (batch, progress bar on) ---
print(f"[2/3] MiniLM embeddings on {DEVICE} (refs/bases/sfts) ...", flush=True)
t0 = time.time()
try:
    from sentence_transformers import SentenceTransformer
    import numpy as np
    enc = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
    print("  encoding refs (1/3) ...", flush=True)
    Er = enc.encode(refs, normalize_embeddings=True, show_progress_bar=True, batch_size=BSZ)
    print("  encoding bases (2/3) ...", flush=True)
    Eb = enc.encode(bases, normalize_embeddings=True, show_progress_bar=True, batch_size=BSZ)
    print("  encoding sfts (3/3) ...", flush=True)
    Es = enc.encode(sfts, normalize_embeddings=True, show_progress_bar=True, batch_size=BSZ)
    cos_base = list(float(np.dot(a,b)) for a,b in zip(Er, Eb))
    cos_sft = list(float(np.dot(a,b)) for a,b in zip(Er, Es))
    print(f"[2/3] embeddings done in {time.time()-t0:.1f}s", flush=True)
except Exception as e:
    print(f"[2/3] embeddings skipped: {e}", flush=True)
    cos_base = [0.0]*len(rows); cos_sft = [0.0]*len(rows)

# --- Stage 3/3: Row-wise lexical + exact-match (CPU, full text) ---
print(f"[3/3] lexical (ROUGE/BLEU/chrF/METEOR, FULL TEXT) on {len(rows)} rows ...", flush=True)
t0 = time.time()
auto_rows = []
for i, r in enumerate(tqdm(rows, desc='lexical', unit='row', dynamic_ncols=True)):
    if i % 10 == 0:
        print(f"  lexical {i}/{len(rows)} ...", flush=True)
    ref, b, s = refs[i], bases[i], sfts[i]
    def lex(hyp):
        sc = rouge.score(ref, hyp)
        bleu = sacrebleu.sentence_bleu(hyp, [ref]).score / 100.0
        chrf = sacrebleu.sentence_chrf(hyp, [ref]).score / 100.0
        return {'rouge1': sc['rouge1'].fmeasure, 'rouge2': sc['rouge2'].fmeasure, 'rougeL': sc['rougeL'].fmeasure,
                'bleu': bleu, 'chrf': chrf, 'meteor': safe_meteor(ref, hyp)}
    lb, ls = lex(b), lex(s)
    tt = (r.get('task_type') or '').lower()
    num_match_b = num_match_s = mcq_match_b = mcq_match_s = None
    if tt in ('numerical','mcq') or tt == 'practice_question':
        rn, bn, sn = num_extract(ref), num_extract(b), num_extract(s)
        if rn is not None:
            num_match_b = (bn is not None and abs(bn-rn) <= 0.01*max(1,abs(rn)))
            num_match_s = (sn is not None and abs(sn-rn) <= 0.01*max(1,abs(rn)))
        if tt == 'mcq':
            rm, bm, sm = mcq_extract(ref), mcq_extract(b), mcq_extract(s)
            mcq_match_b = (bm == rm) if rm else None
            mcq_match_s = (sm == rm) if rm else None
    comp_b = 0.7*bert_base[i] + 0.3*lb['rougeL']
    comp_s = 0.7*bert_sft[i] + 0.3*ls['rougeL']
    d = comp_s - comp_b
    auto_winner = 'tie' if abs(d) < 0.02 else ('sft' if d > 0 else 'base')
    auto_rows.append({'id': r.get('id'), 'auto_base': {**lb, 'bertscore_f1': bert_base[i], 'cosine': cos_base[i], 'composite': comp_b},
                        'auto_sft': {**ls, 'bertscore_f1': bert_sft[i], 'cosine': cos_sft[i], 'composite': comp_s},
                        'auto_delta_sft_minus_base': d, 'auto_winner': auto_winner,
                        'num_match_base': num_match_b, 'num_match_sft': num_match_s,
                        'mcq_match_base': mcq_match_b, 'mcq_match_sft': mcq_match_s})
print(f"[3/3] lexical done in {time.time()-t0:.1f}s", flush=True)
with open(AUTO_OUT, 'w') as f:
    for a in auto_rows:
        f.write(json.dumps(a) + '\n')
print(f"[Part A] saved {AUTO_OUT} n={len(auto_rows)} total {time.time()-t_all:.1f}s", flush=True)
import pandas as pd
pd.DataFrame(auto_rows).head(3)



[Part A] device=cuda bert_batch=64 rows=150
[Part A] starting auto metrics on 150 rows...
[1/3] BERTScore roberta-large on cuda (batch=64): base vs ref ...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

lexical:  92%|█████████▏| 138/150 [00:26<00:00, 14.09row/s]

  lexical 140/150 ...


lexical: 100%|██████████| 150/150 [00:27<00:00,  5.45row/s]

[3/3] lexical done in 27.6s
[Part A] saved /content/eval_auto_scored.jsonl n=150 total 118.5s


,id,auto_base,auto_sft,auto_delta_sft_minus_base,auto_winner,num_match_base,num_match_sft,mcq_match_base,mcq_match_sft
0,eval_001,"{'rouge1': 0.35000000000000003, 'rouge2': 0.10...","{'rouge1': 0.4137931034482759, 'rouge2': 0.195...",0.035780,sft,None,None,None,None
1,eval_002,"{'rouge1': 0.4, 'rouge2': 0.11260053619302948,...","{'rouge1': 0.6639999999999999, 'rouge2': 0.532...",0.185992,sft,False,True,None,None
2,eval_003,"{'rouge1': 0.35967302452316074, 'rouge2': 0.07...","{'rouge1': 0.3873015873015873, 'rouge2': 0.079...",0.008814,tie,False,True,None,None
